In [ ]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import jax_sn
import jax
import jax_sbgeom as jsb
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import pyvista as pv
from typing import Iterable
from jax_sn.domain.cross_sections import CrossSectionData, CrossSectionsIndexed
import numpy as onp 
import matplotlib.pyplot as plt
import equinox as eqx

In [ ]:
vmec_file = ["/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4"][0]
fs_base = jsb.flux_surfaces.FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)

In [ ]:
d_edge_norm = 1.8
flux_surface_extended = jsb.flux_surfaces.create_extended_flux_surface_d_interp_equal_arclength(fs_base, d_edge_norm, 70, 80, 100)

In [ ]:
dim = 2
degree = 4
tn_order = 3
n_elem_regions = (10, 1, 1, 10, 5, 1, 10, 2)
groups_list = (14.1e6, 10e6, 3e6, 1e6, 1e4, 300, 10,0)

d_regions_base = (1.2, 0.002, 0.025, 0.5, 0.385, 0.06, 0.2, 0.06)
l_order = 4
source_kwargs = {
    "n0"    : 2.4e20,
    "Ti0"   : 15.2,
    "nalpha": 0.35,
    "Tialpha": 1.2,
}
element = jax_sn.solution_domain.BasixLagrangianSimplex(degree, dim)

In [ ]:
from breeding_blanket_setup import _get_hcpb_data_openmc
new_bb_data_dict, new_aux_data_dict, tally_result_dict = _get_hcpb_data_openmc(d_regions_base, l_order, 1e5, groups_list)

In [ ]:
phi               = jnp.pi/25
n_theta          = 40

In [ ]:
from breeding_blanket_setup import setup_mgop
MGOP_EA, solution_domain_ea, source_all_angles_ea = setup_mgop(d_regions_base, phi, n_theta, n_elem_regions, tn_order, flux_surface_extended, source_kwargs, new_bb_data_dict, element, normalisation_distance = d_edge_norm)

In [ ]:
source_all_angles_ea.shape

In [ ]:
from jax_sn.operators.multi_group_operator.lower_triangular_solver import MultiGroupLowerTriangularBlockSolver
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import _solve_multigroup_operator_rhs_scan
from breeding_blanket_setup import line_interpolate, plot_domain_with_lines_grid
import lineax as lx
mgsolver = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(rtol = 1e-10, atol=1e-10, max_steps = 250))


In [ ]:
plot_domain_with_lines_grid(solution_domain_ea, solution_total_ea, fs_base, phi, groups_list)

In [ ]:
from jax_sbgeom.jax_utils.splines import bspline, periodic_bspline, greville_abscissa_periodic_bspline

In [ ]:
from typing import Tuple

class BZShield2DParametrisation(eqx.Module):
    c_bz     : jnp.ndarray
    c_shield : jnp.ndarray

    @classmethod
    def from_settings(cls, n_ctrl, settings : "BZShield2DSettings"):

        c_bz = jnp.full(n_ctrl, settings.d_regions_base[3])
        c_shield = jnp.full(n_ctrl,settings.d_regions_base[-2])
        return cls(c_bz, c_shield)

class BZShield2DSettings(eqx.Module):
    k_spline           : int
    n_theta_blanket    : int
    d_regions_base     : Tuple[float]
    n_elem_regions     : Tuple[int]
    phi                : float
    normalise_distance : float
    s_power_sampling   : float 
    flux_surface_extended : jsb.flux_surfaces.FluxSurface    

@eqx.filter_jit
def _get_flux_surface_coordinates(param : BZShield2DParametrisation, settings : BZShield2DSettings):

    theta_single_layer = jnp.linspace(0, 2*jnp.pi, settings.n_theta_blanket, endpoint=False)    
    d_regions_before   = jnp.array([jnp.full_like(theta_single_layer, d) for d in settings.d_regions_base[0:3]])
    d_bz               = periodic_bspline(theta_single_layer, param.c_bz, settings.k_spline)
    d_regions_middle   = jnp.array([jnp.full_like(theta_single_layer, d) for d in settings.d_regions_base[4:-2]])
    d_shield           = periodic_bspline(theta_single_layer, param.c_shield, settings.k_spline)
    d_regions_after    = jnp.array([jnp.full_like(theta_single_layer, d) for d in settings.d_regions_base[-1:]])

    total_d_array = jnp.concatenate([d_regions_before, d_bz[None, :], d_regions_middle, d_shield[None, :], d_regions_after], axis=0) #[n_regions, n_theta]

    total_d_array_cumsum = jnp.concatenate([jnp.zeros((1,n_theta)), jnp.cumsum(total_d_array, axis=0)], axis=0)  # [n_regions, n_theta]

    def _single_theta_linspaces(i):
        s_from_edge = jnp.concatenate([jnp.linspace(total_d_array_cumsum[j, i], total_d_array_cumsum[j+1, i], n_elem_regions[j], endpoint=False) for j in range(0, len(settings.d_regions_base))] + [jnp.array([total_d_array_cumsum[-1, i]])])

        return jnp.where(s_from_edge < 1.0, s_from_edge**settings.s_power_sampling, 1.0 + (s_from_edge - 1.0) / (settings.normalise_distance))[1:] # do not include the axis!

    d_total_array = jax.vmap(_single_theta_linspaces, in_axes=(0))(jnp.arange(settings.n_theta_blanket))
    theta_total = jnp.meshgrid(theta_single_layer, jnp.arange(d_total_array.shape[1]), indexing='ij')[0]  # [n_theta, n_regions*n_elem_regions]

    return theta_total.T, jnp.full_like(theta_total, settings.phi).T, d_total_array.T
    
@eqx.filter_jit
def _get_mesh_points(param : BZShield2DParametrisation, settings : BZShield2DSettings):

    theta_total, phi_total, d_total_array = _get_flux_surface_coordinates(param, settings)

    

    return settings.flux_surface_extended.cylindrical_position(jnp.concatenate([jnp.array([0]), d_total_array.ravel()]), 
                                                               jnp.concatenate([jnp.array([0]), theta_total.ravel()]),
                                                               jnp.concatenate([jnp.array([phi_total[0,0]]), phi_total.ravel()]))[..., :2].reshape(-1,2)

settings = BZShield2DSettings(
    k_spline = 3, 
    n_theta_blanket = n_theta,
    d_regions_base = d_regions_base,
    n_elem_regions = n_elem_regions,
    phi = phi,
    normalise_distance = d_edge_norm,
    s_power_sampling= 2,
    flux_surface_extended = flux_surface_extended    
)
n_ctrl = 16
param = BZShield2DParametrisation.from_settings(n_ctrl, settings)
coordinates = _get_mesh_points(param, settings)

In [ ]:
onp.testing.assert_allclose(MGOP_EA.transport_operator.solution_domain_data.domain.vertices, coordinates)

In [ ]:
from jax_sn.tools.parametrisation_tools import ParametrisedModuleTheta
from jax_sn.parametrisation.multigroup_parametrisation import MultigroupOperatorParametrisedVertices, MultigroupOperatorParametrisedVerticesNoToposort, MultigroupOperatorParametrisedObjective
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import MultiGroupOperator

class BZShield2DMGOP(ParametrisedModuleTheta):
    additional_settings : BZShield2DSettings

    def __check_init__(self):
        assert isinstance(self.additional_settings, BZShield2DSettings)
        assert isinstance(self.base_module, MultiGroupOperator)

    @eqx.filter_jit
    def to_module(self, theta : BZShield2DParametrisation):        
        return MultigroupOperatorParametrisedVertices.reconstruct_operator(self.base_module, _get_mesh_points(theta, self.additional_settings))
    
    @eqx.filter_jit
    def to_module_no_toposort(self, theta : BZShield2DParametrisation):        
        return MultigroupOperatorParametrisedVerticesNoToposort.reconstruct_operator(self.base_module, _get_mesh_points(theta, self.additional_settings))
    
class BZShield2DMultiGroupObjective(MultigroupOperatorParametrisedObjective):

    @eqx.filter_jit
    def objective_and_constraints(self, theta : BZShield2DParametrisation):
        return jnp.sum(self.solution(theta).value)
    
mgsolver = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(rtol = 1e-10, atol=1e-10, max_steps = 250))
mgop_parametrised = BZShield2DMGOP(MGOP_EA, additional_settings = settings)

mgop_objective = BZShield2DMultiGroupObjective(mgop_parametrised, source_all_angles_ea, mgsolver)

def obj(theta, multigroup_objective):
    return multigroup_objective.objective_and_constraints(theta)


In [ ]:
mgop_objective.objective_and_constraints(param)

In [ ]:
jax.config.update("jax_log_compiles", True)

In [ ]:
f_grad = eqx.filter_jit(jax.grad(obj, argnums =(0,)))

In [ ]:
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
_filter_jit_f_to_memory(eqx.filter_jit(obj), param, mgop_objective)

In [ ]:
f_grad(param_2, mgop_objective)

In [ ]:
ordered_nt_xs_names = ['Plasma', "TungstenArmor", "FirstWall", "BreedingZone", "BackSupportingStructure","VVWallFront", "VVShield", "VVWallBack"]

In [ ]:
new_aux_data_dict 

In [ ]:
new_aux_data_dict['BreedingZone']

In [ ]:
mgsolver = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(rtol = 1e-10, atol=1e-10, max_steps = 250))
mgop_parametrised = BZShield2DMGOP(MGOP_EA, additional_settings = settings)

mgop_objective = BZShield2DMultiGroupObjective(mgop_parametrised, source_all_angles_ea, mgsolver)

In [ ]:
param_2 = eqx.tree_at(lambda p: p.c_bz, param, replace_fn = lambda c: c.at[0].set(0.2).at[10].set(1.1))

In [ ]:
solution_0 = mgop_objective.solution(param)

In [ ]:
solution_1 = mgop_objective.solution(param_2)

In [ ]:
def get_solution_domain(param, mgop_objective):
    return jax_sn.solution_domain.SolutionDomain(mgop_objective.to_module_no_toposort(param).transport_operator.solution_domain_data, element)


get_solution_domain(param, mgop_objective)

In [ ]:
plot_domain_with_lines_grid(get_solution_domain(param, mgop_objective), solution_0, fs_base, phi, groups_list)

In [ ]:
plot_domain_with_lines_grid(get_solution_domain(param_2, mgop_objective), solution_1, fs_base, phi, groups_list, n_lines = 6, grid_cols =3 )

In [ ]:
from breeding_blanket_setup import plot_domain, plot_domain_with_lines_grid

plot_domain(mgop_objective.to_module(param).domain)
plot_domain(mgop_objective.to_module(param_2).domain)

In [ ]:
plot_domain(mgop_parametrised.to_module_no_toposort(param_2).domain)

In [ ]:
from breeding_blanket_setup import plot_periodic_spline_with_controls


plot_periodic_spline_with_controls(param_2.c_bz, settings.k_spline, True)

In [ ]:
flatten_tree_recon = jax.tree_util.tree_flatten(mgop_parametrised.to_module(param))

In [ ]:
from jax_sn.tools.memory_analysis import assert_pytree_allclose

In [ ]:
assert_pytree_allclose(mgop_parametrised.to_module(param), MGOP_EA, atol =1e-9, rtol = 1e-9)

In [ ]:
flatten_tree_recon[0] == jax.tree_util.tree_flatten( MGOP_EA)

In [ ]:
MGOP_EA.domain

In [ ]:
onp.testing.assert_allclose(MGOP_EA.transport_operator.solution_domain_data.domain.vertices[1:], coordinates)

In [ ]:

 

# def _get_values(vertices_base : jnp.ndarray, ):
#     import jax_sbgeom as jsb
#     s_power_sampling = 2
#     regions_dim = [(sz, res) for sz, res in zip(d_regions_base, n_elem_regions)]
#     cumsum_nodes  = jnp.concatenate([jnp.array([0]), jnp.cumsum(jnp.array([sz for sz, res in regions_dim]))])        

#     s_base = [jnp.linspace(cumsum_nodes[0], cumsum_nodes[1], n_elem_regions[0], endpoint=False)**s_power_sampling]
#     s_from_edge = [jnp.linspace(cumsum_nodes[i], cumsum_nodes[i+1], n_elem_regions[i], endpoint=False) for i in range(1, len(regions_dim))] + [jnp.array([cumsum_nodes[-1]])]
#     if normalise_distance is not None:
#         assert isinstance(normalise_distance, float), f"normalise_distance should be a float but is {type(normalise_distance)}"
#         assert normalise_distance > cumsum_nodes[-1] - 1, f"normalise_distance ({normalise_distance}) should be larger than the total size of the blanket ({cumsum_nodes[-1] - 1})"        
#         s_array_base = jnp.concatenate(s_base + s_from_edge)
#         s_array = jnp.where(s_array_base < 1.0, s_array_base, 1.0 + (s_array_base - 1.0) / (normalise_distance ))
#     else:
#         s_array = jnp.concatenate(s_base + s_from_edge)

#     mesh_pol = jsb.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(flux_surface_extended,s_array, phi, n_theta )
#     print(mesh_pol[0])


# _get_values(d_regions_base, n_elem_regions, flux_surface_extended, n_theta, phi, source_kwargs, normalise_distance = d_edge_norm)